# Steps 7 and 9: Faithfulness checker and test-set evaluation on Colab

**Author:** Michael Dong  
**Project:** PlainMed (COSC 72/272 final project)  
**Date:** November 15, 2026

**What this notebook does**
1. Copies the Step 4 validation outputs from Google Drive into the project.
2. Checks that the prompt chosen on validation is frozen in `config.yaml` (CONTRACT rule 1: the test set is used once, after every choice is fixed).
3. Runs both LLM systems on the **Cochrane test set** (480 examples each) with the frozen prompt.
4. Writes the baseline outputs for the test set.
5. Runs the **faithfulness checker** (DeBERTa-v3 NLI + number rule) on every system's validation and test outputs, and on the six prompt-sensitivity runs.
6. Scores every system with all metrics, including **real BERTScore**, with 95% confidence intervals.

**Input:** the GitHub repo and `MyDrive/plainmed/step4/` from the Step 4 notebook.  
**Output:** `MyDrive/plainmed/step9/` (test outputs as they are generated, then the checked outputs and all results at the end).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. The `GH_TOKEN` secret from Step 4 is reused.

### 1. GPU, Drive and code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
STEP4 = '/content/drive/MyDrive/plainmed/step4'
RUN_DIR = '/content/drive/MyDrive/plainmed/step9'
os.makedirs(RUN_DIR, exist_ok=True)
if not os.path.exists('/content/plainmed'):
    if subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/dongm58/plainmed', '/content/plainmed']).returncode != 0:
        from google.colab import userdata
        subprocess.run(['git', 'clone', '-q', '--depth', '1', f"https://{userdata.get('GH_TOKEN')}@github.com/dongm58/plainmed", '/content/plainmed'], check=True)
%cd /content/plainmed
!git log -1 --format='commit %h: %s'

### 2. Install and build the data

In [ ]:
!pip -q install -r requirements-colab.txt && pip -q install -e . --no-deps
!python scripts/01_data.py > /tmp/data_log.txt && head -2 /tmp/data_log.txt

### 3. Bring in the Step 4 outputs and check the frozen prompt

In [ ]:
!mkdir -p outputs && cp -r $STEP4/outputs/llm_zeroshot $STEP4/outputs/llm_fewshot $STEP4/outputs/prompt_sensitivity outputs/
!wc -l outputs/llm_zeroshot/cochrane_val.jsonl outputs/llm_fewshot/cochrane_val.jsonl
import json, yaml
config = yaml.safe_load(open('config.yaml'))
choice = json.load(open(f'{STEP4}/results/prompt_choice.json'))['chosen_variant']
VARIANT = config['llm']['prompt_variant']
assert VARIANT == choice, f'config.yaml has {VARIANT!r} but Step 4 chose {choice!r}: ask Claude to freeze the prompt first'
print('frozen prompt variant:', VARIANT)

### 4. LLM systems on the Cochrane test set (480 each, about 15–20 minutes each)
Saved to Drive as they go. After a disconnect, run cells 1–3 again, then this cell: finished examples are skipped.

In [ ]:
!python scripts/04_llm.py run --shots zero --variant $VARIANT --dataset cochrane --split test --out-root $RUN_DIR --batch-size 8

In [ ]:
!python scripts/04_llm.py run --shots few --variant $VARIANT --dataset cochrane --split test --out-root $RUN_DIR --batch-size 8

In [ ]:
!cp -r $RUN_DIR/outputs/llm_zeroshot/cochrane_test.jsonl outputs/llm_zeroshot/ && cp -r $RUN_DIR/outputs/llm_fewshot/cochrane_test.jsonl outputs/llm_fewshot/
!python scripts/03_baselines.py --split test > /tmp/baselines_log.txt && tail -1 /tmp/baselines_log.txt
!wc -l outputs/*/cochrane_test.jsonl

### 5. Faithfulness checker on everything (downloads the NLI model, about 0.7 GB)

In [ ]:
import glob
FILES = ' '.join([f'outputs/{s}/cochrane_{sp}.jsonl' for s in ['llm_zeroshot', 'llm_fewshot', 'extractive', 'reference'] for sp in ['val', 'test']] + sorted(glob.glob('outputs/prompt_sensitivity/*.jsonl')))
print(FILES)
!python scripts/07_check.py --batch-size 64 --inputs $FILES

### 6. All metrics with real BERTScore (downloads roberta-large, about 1.4 GB)

In [ ]:
for split in ['val', 'test']:
    for system in ['copy_source', 'extractive', 'reference', 'llm_zeroshot', 'llm_fewshot']:
        !python scripts/02_score.py --system {system} --dataset cochrane --split {split} --bertscore real

### 7. Save everything to Drive
Tell Claude when this finishes.

In [ ]:
!mkdir -p $RUN_DIR/final && cp -r outputs results checked $RUN_DIR/final/
!ls $RUN_DIR/final/results
import pandas as pd
print(pd.read_csv('results/checker_summary.csv')[['file', 'n_outputs', 'share_sentences_flagged', 'share_sentences_number_mismatch']].round(3).to_string(index=False))